# 03 — Créer un environnement personnalisé avec Gymnasium

Jusqu'ici on utilisait des environnements existants (LunarLander). Ce notebook montre comment créer le sien.

Source : https://pythonprogramming.net/custom-environment-reinforcement-learning-stable-baselines-3-tutorial/

## Pourquoi créer un environnement custom ?

Les environnements gymnasium couvrent des cas classiques (jeux, robotique…), mais en pratique on veut souvent appliquer le RL à des problèmes métier :
- Optimiser un portefeuille financier
- Contrôler un robot industriel
- Gérer des ressources dans un datacenter

Dans tous ces cas, il faut **définir son propre environnement**.

## Structure obligatoire d'un environnement gymnasium

Tout environnement doit hériter de `gymnasium.Env` et implémenter :

```
class MonEnv(gymnasium.Env):
    def __init__(self)       → définir observation_space et action_space
    def reset(self)          → réinitialiser et retourner (obs, info)
    def step(self, action)   → appliquer l'action, retourner (obs, reward, terminated, truncated, info)
    def render(self)         → afficher l'état (optionnel)
    def close(self)          → nettoyer les ressources (optionnel)
```

C'est comme une **interface** : SB3 s'attend à ces méthodes, peu importe ce que fait l'environnement.

## Exemple : environnement de navigation 2D simple

Un agent se déplace sur une grille 10×10 et doit atteindre une cible.

- **Observation** : position de l'agent + position de la cible
- **Action** : 4 directions (haut, bas, gauche, droite)
- **Reward** : +10 si cible atteinte, -0.1 à chaque step (encourage l'efficacité)
- **Terminaison** : cible atteinte ou 200 steps dépassés

In [ ]:
import numpy as np
import gymnasium as gym
from gymnasium import spaces


class GridNavigationEnv(gym.Env):
    """Environnement de navigation sur une grille SIZE×SIZE."""

    metadata = {'render_modes': ['human', 'ansi']}

    SIZE = 10  # taille de la grille
    MAX_STEPS = 200

    def __init__(self, render_mode=None):
        super().__init__()
        self.render_mode = render_mode

        # Espace d'action : 4 directions
        # Discrete(N) → entiers de 0 à N-1
        self.action_space = spaces.Discrete(4)
        # 0 = haut, 1 = bas, 2 = gauche, 3 = droite

        # Espace d'observation : [agent_x, agent_y, cible_x, cible_y]
        # Box(low, high, shape, dtype) → vecteur de valeurs continues
        self.observation_space = spaces.Box(
            low=0,
            high=self.SIZE - 1,
            shape=(4,),
            dtype=np.float32
        )

        self._step_count = 0

    def reset(self, seed=None, options=None):
        # seed permet la reproductibilité (bonne pratique)
        super().reset(seed=seed)

        # Placer agent et cible aléatoirement (sans se superposer)
        self.agent_pos = np.array(
            self.np_random.integers(0, self.SIZE, size=2),
            dtype=np.float32
        )
        self.target_pos = np.array(
            self.np_random.integers(0, self.SIZE, size=2),
            dtype=np.float32
        )
        # Garantir que agent != cible
        while np.array_equal(self.agent_pos, self.target_pos):
            self.target_pos = np.array(
                self.np_random.integers(0, self.SIZE, size=2),
                dtype=np.float32
            )

        self._step_count = 0

        obs = self._get_obs()
        info = {}
        return obs, info

    def step(self, action):
        # Appliquer le mouvement
        moves = {
            0: np.array([0, 1]),   # haut
            1: np.array([0, -1]),  # bas
            2: np.array([-1, 0]),  # gauche
            3: np.array([1, 0]),   # droite
        }
        self.agent_pos = np.clip(
            self.agent_pos + moves[action],
            0, self.SIZE - 1
        ).astype(np.float32)

        self._step_count += 1

        # Calculer la récompense
        reached = np.array_equal(self.agent_pos, self.target_pos)
        reward = 10.0 if reached else -0.1  # pénalité par step = encourage la rapidité

        # Conditions de fin
        terminated = reached              # succès
        truncated = self._step_count >= self.MAX_STEPS  # timeout

        obs = self._get_obs()
        info = {"step": self._step_count}

        return obs, reward, terminated, truncated, info

    def render(self):
        if self.render_mode == 'ansi':
            grid = [['.' for _ in range(self.SIZE)] for _ in range(self.SIZE)]
            ax, ay = int(self.agent_pos[0]), int(self.agent_pos[1])
            tx, ty = int(self.target_pos[0]), int(self.target_pos[1])
            grid[ay][ax] = 'A'
            grid[ty][tx] = 'T'
            print('\n'.join(''.join(row) for row in reversed(grid)))
            print()

    def close(self):
        pass

    def _get_obs(self):
        return np.concatenate([self.agent_pos, self.target_pos])


print("Environnement GridNavigationEnv défini")

## 1. Tester l'environnement manuellement

Avant d'entraîner, vérifier que l'environnement se comporte correctement.

In [ ]:
env = GridNavigationEnv(render_mode='ansi')
obs, info = env.reset(seed=42)

print("Observation initiale :")
print(f"  Agent   : ({obs[0]:.0f}, {obs[1]:.0f})")
print(f"  Cible   : ({obs[2]:.0f}, {obs[3]:.0f})")
print()

env.render()

# Faire 3 actions et observer
for action_name, action in [("droite", 3), ("haut", 0), ("droite", 3)]:
    obs, reward, terminated, truncated, info = env.step(action)
    print(f"Action '{action_name}' → reward={reward:.1f}, terminated={terminated}")

env.close()

## 2. Valider avec le checker SB3

SB3 fournit un outil qui vérifie automatiquement que l'environnement est conforme à l'API gymnasium.

In [ ]:
from stable_baselines3.common.env_checker import check_env

env = GridNavigationEnv()
check_env(env, warn=True)
print("Environnement valide !")
env.close()

## 3. Entraîner un agent sur l'environnement custom

In [ ]:
import os
from stable_baselines3 import PPO
from stable_baselines3.common.evaluation import evaluate_policy

os.makedirs("models/grid_nav", exist_ok=True)
os.makedirs("logs", exist_ok=True)

env = GridNavigationEnv()
model = PPO('MlpPolicy', env, verbose=1, tensorboard_log="logs")

TIMESTEPS = 10_000
for i in range(1, 6):
    model.learn(total_timesteps=TIMESTEPS, reset_num_timesteps=False, tb_log_name="GridNav")
    model.save(f"models/grid_nav/{TIMESTEPS * i}")
    print(f"Checkpoint {TIMESTEPS * i} sauvegardé")

env.close()

In [ ]:
env = GridNavigationEnv()
mean_reward, std_reward = evaluate_policy(model, env, n_eval_episodes=20)
print(f"Reward moyen : {mean_reward:.2f} ± {std_reward:.2f}")
env.close()

## 4. Concevoir les observations : c'est le vrai défi

> *"La partie la plus difficile du RL est de concevoir les observations et les récompenses de l'environnement."*

Exemples de choix d'observation pour le Snake :

| Approche | Avantage | Inconvénient |
|----------|----------|--------------|
| Image brute (pixels) | Aucune information perdue | Lent, nécessite CNN |
| Vecteur compact (`[head_x, head_y, dx_pomme, dy_pomme, longueur]`) | Rapide, MlpPolicy suffit | Nécessite un choix pertinent |
| Distance à l'objectif | Simple et efficace | Perd la structure spatiale |

Pour notre grille, on a choisi : `[agent_x, agent_y, cible_x, cible_y]`

Alternative : donner directement le **vecteur delta** (différence de position) :

In [ ]:
# Exemple de variante avec vecteur delta
class GridNavDeltaEnv(GridNavigationEnv):
    """Variante où l'observation est le vecteur delta (cible - agent)."""

    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        # Redéfinir l'espace d'observation : delta peut être négatif
        self.observation_space = spaces.Box(
            low=-(self.SIZE - 1),
            high=self.SIZE - 1,
            shape=(2,),  # seulement dx, dy
            dtype=np.float32
        )

    def _get_obs(self):
        # L'agent reçoit directement la direction vers la cible
        return (self.target_pos - self.agent_pos).astype(np.float32)


# Valider et tester
env_delta = GridNavDeltaEnv()
check_env(env_delta)
obs, _ = env_delta.reset(seed=0)
print(f"Observation delta : {obs}  (dx={obs[0]:.0f}, dy={obs[1]:.0f})")
env_delta.close()

## Points clés à retenir

1. **Hériter de `gym.Env`** et implémenter `__init__`, `reset`, `step`
2. **Définir `observation_space` et `action_space`** dans `__init__` — SB3 en a besoin
3. **`reset()` retourne `(obs, info)`** — ne pas oublier `info`
4. **`step()` retourne `(obs, reward, terminated, truncated, info)`** — `terminated` ≠ `truncated`
5. **Utiliser `check_env()`** avant d'entraîner — évite des bugs cryptiques
6. **La conception des observations est cruciale** — le réseau ne peut apprendre que ce qu'il voit

**Prochaine étape** : concevoir les récompenses → notebook `04_reward_engineering.ipynb`